# 18 · Feature Extraction  (`18_nonvlm_fe`)

Two extractor classes that share the same **locked 56-column feature schema**
so their outputs can be directly compared:

| Class | Method | Cost | Speed |
|---|---|---|---|
| `HaikuExtractor` | Claude Haiku VLM | ~$0.02/card | slow (API) |
| `CVExtractor` | Classical CV (YOLO + OpenCV) | free | ~0.3 s/card |

**Already cached** — existing CSVs are ready to use:
- `feature_extraction_dataset/haiku_features.csv` (694 cards)
- `feature_extraction_dataset/cv_features.csv` (679 cards)

Run the extractor classes to process **new** images or re-extract from scratch.

In [ ]:
import os, sys, json, glob, importlib
import numpy as np, pandas as pd, cv2
from pathlib import Path
from dotenv import load_dotenv

os.environ["CARD_DETECTOR"] = "yolo"
os.environ.setdefault("YOLO_WEIGHTS", "../backend/models/yolo_obb_best.pt")
sys.path.insert(0, "../backend"); sys.path.insert(0, ".")
load_dotenv("../.env.local", override=True); load_dotenv("../backend/.env", override=False)

import nonvlm_cv as N
import grader; importlib.reload(grader)

BASE   = Path("feature_extraction_dataset")
MODELS = Path("models"); MODELS.mkdir(exist_ok=True)
print("ANTHROPIC key:", bool(os.environ.get("ANTHROPIC_API_KEY")))
print("Cached CSVs:")
for f in ["haiku_features.csv", "cv_features.csv"]:
    p = BASE / f
    if p.exists():
        df = pd.read_csv(p)
        print(f"  {f}: {len(df)} rows × {len(df.columns)} cols")
    else:
        print(f"  {f}: NOT FOUND")

## HaikuExtractor

In [ ]:
class HaikuExtractor:
    """Extract card condition features using Claude Haiku.

    Output CSV columns:
      file, actual_psa, path, is_sir,
      cen.lr_deviation, cen.tb_deviation,
      <56 VLM feature columns>,
      _raw_features, error

    Usage (new images):
        ext = HaikuExtractor()
        ext.extract_folder("feature_extraction_dataset/5", psa_grade=5)
        ext.extract_folder("feature_extraction_dataset/10", psa_grade=10)
        ext.save("feature_extraction_dataset/haiku_features.csv")

    Usage (load existing cache):
        ext = HaikuExtractor.from_csv("feature_extraction_dataset/haiku_features.csv")
        df  = ext.to_dataframe()
    """

    OUTPUT_PREFIX = "haiku"
    MODEL         = "claude-haiku-4-5"

    def __init__(self, api_key: str = None, detector: str = "seg"):
        """
        Args:
            api_key:  Anthropic API key (defaults to ANTHROPIC_API_KEY env var)
            detector: "seg" (Roboflow, matches existing cache) or "yolo" (free, local)
        """
        self.api_key  = api_key or os.environ.get("ANTHROPIC_API_KEY", "")
        self.detector = detector
        self._rows    = []

    # ── public API ──────────────────────────────────────────────────────────

    def extract_card(self, image_path: str, psa_grade: int,
                     is_sir: bool = False) -> dict:
        """Extract Haiku features for one card. Returns a row dict."""
        img = cv2.imread(image_path)
        if img is None:
            return {"file": os.path.basename(image_path), "actual_psa": psa_grade,
                    "path": os.path.abspath(image_path), "error": "Cannot read image"}

        det  = N.detect_and_warp(img)
        cen  = N.compute_centering_hybrid(det["warped"], det["cb"])
        cr   = cen["content_region"]
        cond = N.grade_conditions(det, api_key=self.api_key)

        if "error" in cond:
            return {"file": os.path.basename(image_path), "actual_psa": psa_grade,
                    "path": os.path.abspath(image_path), "error": str(cond["error"])[:120]}

        vec  = N.features_to_vector(cond)
        lr   = int(cen["left_right"].split("/")[0])
        tb   = int(cen["top_bottom"].split("/")[0])

        return {"file": os.path.basename(image_path), "actual_psa": psa_grade,
                "path": os.path.abspath(image_path), "is_sir": is_sir,
                "cen.lr_deviation": abs(lr - 50),
                "cen.tb_deviation":  abs(tb - 50),
                **vec, "_raw_features": json.dumps(cond)}

    def extract_folder(self, folder: str, psa_grade: int,
                       pattern: str = "*_front.jpeg",
                       resume_csv: str = None,
                       verbose: bool = True) -> "HaikuExtractor":
        """Extract all images matching pattern in folder.

        Args:
            folder:     path to image directory
            psa_grade:  integer PSA grade for all images in this folder
            pattern:    glob pattern for images (default: *_front.jpeg)
            resume_csv: existing CSV to skip already-processed paths
            verbose:    print progress
        """
        done_paths = set()
        if resume_csv and os.path.exists(resume_csv):
            df_done = pd.read_csv(resume_csv)
            done_paths = set(df_done["path"].dropna().apply(os.path.abspath))
            print(f"Resuming: {len(done_paths)} already done")

        files = sorted(glob.glob(f"{folder}/{pattern}"))
        todo  = [f for f in files if os.path.abspath(f) not in done_paths]
        print(f"[PSA {psa_grade}] {len(todo)}/{len(files)} to extract")

        for i, path in enumerate(todo, 1):
            if verbose:
                print(f"  ({i}/{len(todo)}) {os.path.basename(path)} ...", end=" ", flush=True)
            row = self.extract_card(path, psa_grade)
            if verbose:
                status = f"ERROR: {row.get('error','')[:60]}" if "error" in row else "ok"
                print(status)
            self._rows.append(row)
        return self

    def extract_all_grades(self, base_dir: str,
                           grades: tuple = (5, 6, 7, 8, 9, 10),
                           resume_csv: str = None) -> "HaikuExtractor":
        """Extract every grade subfolder under base_dir."""
        for g in grades:
            folder = os.path.join(base_dir, str(g))
            if os.path.isdir(folder):
                self.extract_folder(folder, psa_grade=g, resume_csv=resume_csv)
        return self

    def to_dataframe(self) -> pd.DataFrame:
        """Return all accumulated rows as a DataFrame."""
        return pd.DataFrame(self._rows) if self._rows else pd.DataFrame()

    def save(self, output_path: str = None) -> str:
        """Save to CSV. Defaults to haiku_features.csv in feature_extraction_dataset/."""
        if output_path is None:
            output_path = os.path.join("feature_extraction_dataset", "haiku_features.csv")
        df = self.to_dataframe()
        df.to_csv(output_path, index=False)
        print(f"Saved {len(df)} rows → {output_path}")
        return output_path

    @classmethod
    def from_csv(cls, csv_path: str) -> "HaikuExtractor":
        """Load an existing feature CSV (no re-extraction needed)."""
        inst = cls()
        df   = pd.read_csv(csv_path)
        inst._rows = df.to_dict("records")
        print(f"Loaded {len(inst._rows)} rows from {csv_path}")
        return inst

print("HaikuExtractor defined")

## CVExtractor

In [ ]:
class CVExtractor:
    """Extract card condition features using classical computer vision.

    Output CSV columns:
      file, actual_psa, path, is_sir, detector, seg_conf,
      cen.lr_deviation, cen.tb_deviation,
      cv.<feature_name> × 56

    Usage (new images):
        ext = CVExtractor()
        ext.extract_folder("feature_extraction_dataset/5", psa_grade=5)
        ext.save("feature_extraction_dataset/cv_features.csv")

    Usage (load existing cache):
        ext = CVExtractor.from_csv("feature_extraction_dataset/cv_features.csv")
        df  = ext.to_dataframe()
    """

    OUTPUT_PREFIX = "cv"

    def __init__(self, detector: str = "seg"):
        """
        Args:
            detector: "seg" (Roboflow Model C — production, detects card-in-slab, recommended)
        """
        os.environ["CARD_DETECTOR"] = detector
        self.detector = detector
        self._rows    = []

    # ── public API ──────────────────────────────────────────────────────────

    def extract_card(self, image_path: str, psa_grade: int,
                     is_sir: bool = False) -> dict:
        """Extract CV features for one card. Returns a row dict."""
        img = cv2.imread(image_path)
        if img is None:
            return {"file": os.path.basename(image_path), "actual_psa": psa_grade,
                    "path": os.path.abspath(image_path), "error": "Cannot read image"}
        try:
            det  = N.detect_and_warp(img, detector=self.detector, out_size=N.CV_WARP_SIZE)
            cen  = N.compute_centering_hybrid(det["warped"], det["cb"])
            cr   = cen["content_region"]
            cond, _ = N.cv_extract_conditions(det, cr=cr)
            vec  = N.features_to_vector(cond)
            lr   = int(cen["left_right"].split("/")[0])
            tb   = int(cen["top_bottom"].split("/")[0])
            return {"file": os.path.basename(image_path), "actual_psa": psa_grade,
                    "path": os.path.abspath(image_path), "is_sir": is_sir,
                    "detector": det.get("detector","yolo"),
                    "seg_conf": round(float(det.get("seg_conf", 0)), 3),
                    "cen.lr_deviation": abs(lr - 50),
                    "cen.tb_deviation":  abs(tb - 50),
                    **{f"cv.{k}": v for k, v in vec.items()}}
        except Exception as exc:
            return {"file": os.path.basename(image_path), "actual_psa": psa_grade,
                    "path": os.path.abspath(image_path),
                    "error": f"{type(exc).__name__}: {str(exc)[:120]}"}

    def extract_folder(self, folder: str, psa_grade: int,
                       pattern: str = "*_front.jpeg",
                       resume_csv: str = None,
                       verbose: bool = True) -> "CVExtractor":
        """Extract all images matching pattern in folder."""
        done_paths = set()
        if resume_csv and os.path.exists(resume_csv):
            df_done = pd.read_csv(resume_csv)
            done_paths = set(df_done["path"].dropna().apply(os.path.abspath))
            print(f"Resuming: {len(done_paths)} already done")

        files = sorted(glob.glob(f"{folder}/{pattern}"))
        todo  = [f for f in files if os.path.abspath(f) not in done_paths]
        print(f"[PSA {psa_grade}] {len(todo)}/{len(files)} to extract")

        for i, path in enumerate(todo, 1):
            if verbose:
                print(f"  ({i}/{len(todo)}) {os.path.basename(path)} ...", end=" ", flush=True)
            row = self.extract_card(path, psa_grade)
            if verbose:
                if "error" in row:
                    print(f"ERROR: {row['error'][:60]}")
                else:
                    print(f"ok  cor={row['cv.corners.max']:.0f} "
                          f"edg={row['cv.edges.max']:.0f} "
                          f"srf={row['cv.surface.max']:.0f}  "
                          f"cen={row['cen.lr_deviation']:.0f}/{row['cen.tb_deviation']:.0f}")
            self._rows.append(row)
        return self

    def extract_all_grades(self, base_dir: str,
                           grades: tuple = (5, 6, 7, 8, 9, 10),
                           resume_csv: str = None) -> "CVExtractor":
        """Extract every grade subfolder under base_dir."""
        for g in grades:
            folder = os.path.join(base_dir, str(g))
            if os.path.isdir(folder):
                self.extract_folder(folder, psa_grade=g, resume_csv=resume_csv)
        return self

    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame(self._rows) if self._rows else pd.DataFrame()

    def save(self, output_path: str = None) -> str:
        if output_path is None:
            output_path = os.path.join("feature_extraction_dataset", "cv_features.csv")
        df = self.to_dataframe()
        df.to_csv(output_path, index=False)
        print(f"Saved {len(df)} rows → {output_path}")
        return output_path

    @classmethod
    def from_csv(cls, csv_path: str) -> "CVExtractor":
        inst = cls()
        df   = pd.read_csv(csv_path)
        inst._rows = df.to_dict("records")
        print(f"Loaded {len(inst._rows)} rows from {csv_path}")
        return inst

print("CVExtractor defined")

## Quick demo — load cached CSVs

In [ ]:
# Load existing cached extractions (no re-extraction needed)
haiku_ext = HaikuExtractor.from_csv(BASE / "haiku_features.csv")
cv_ext    = CVExtractor.from_csv(BASE / "cv_features.csv")

df_haiku = haiku_ext.to_dataframe()
df_cv    = cv_ext.to_dataframe()

print("\nHaiku features:")
print(df_haiku.groupby("actual_psa").size().rename("cards").to_frame().T.to_string())
print(f"  columns: {len(df_haiku.columns)}  (56 VLM features + metadata)")

print("\nCV features:")
print(df_cv.groupby("actual_psa").size().rename("cards").to_frame().T.to_string())
print(f"  columns: {len(df_cv.columns)}  (56 CV features + metadata)")

## Extract NEW images (example)

Only run this cell when you have **new card images** to add to the dataset.
The `resume_csv` argument skips cards already in the existing CSV.

```python
# Haiku (costs API credits — ~$0.02/card)
ext = HaikuExtractor()
ext.extract_folder("my_new_cards/psa9", psa_grade=9,
                   resume_csv=str(BASE/"haiku_features.csv"))
ext.save(str(BASE/"haiku_features.csv"))

# CV (free)
ext = CVExtractor()
ext.extract_folder("my_new_cards/psa9", psa_grade=9,
                   resume_csv=str(BASE/"cv_features.csv"))
ext.save(str(BASE/"cv_features.csv"))
```